In [0]:
import importlib
import configs.constant as constants
import utils.transformation_function as transformation_functions
importlib.reload(constants)
importlib.reload(transformation_functions)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from configs.constant import * 

df_silver = spark.read.format("delta") \
    .load(f"abfss://{SILVER_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/clean_data/")

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_ACCESS_KEY
)

In [0]:
try:
    last_processed = spark.read.format("delta") \
        .load(f"abfss://{GOLD_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/fct_events/") \
        .agg(max("event_timestamp")) \
        .collect()[0][0]
except:
    last_processed = None

In [0]:
if last_processed:
    df_incremental = df_silver.filter(col("event_timestamp") > last_processed)
else:
    df_incremental = df_silver

In [0]:
dim_product = df_incremental.select(
    "product_id",
    "category"
).dropDuplicates()

In [0]:
dim_user = df_incremental.select(
    "user_id",
    "region",
    "device_type"
).dropDuplicates()

In [0]:
dim_date = df_incremental.select(
    col("event_date"),
    col("event_timestamp"),
    col("event_hour"),
    weekofyear("event_timestamp").alias("week")
).dropDuplicates(["event_date"])

In [0]:
dim_region = df_incremental.select(
    "region"
).dropDuplicates()

In [0]:
from pyspark.sql.window import Window

window_spec = Window.orderBy("product_id")

dim_product = dim_product.withColumn(
    "product_key",
    row_number().over(window_spec)
)

In [0]:
window_spec = Window.orderBy("user_id")

dim_user = dim_user.withColumn(
    "user_key",
    row_number().over(window_spec)
)

In [0]:
window_spec = Window.orderBy("event_date")

dim_date = dim_date.withColumn(
    "date_key",
    row_number().over(window_spec)
)

In [0]:
window_spec = Window.orderBy("region")

dim_region = dim_region.withColumn(
    "region_key",
    row_number().over(window_spec)
)

In [0]:
fct_events = df_incremental \
    .join(dim_product, "product_id", "left") \
    .join(dim_user, "user_id", "left") \
    .join(dim_date, "event_date", "left") \
    .join(dim_region, "region", "left") \
    .select(
        "event_id",
        "product_key",
        "user_key",
        "date_key",
        "region_key",
        "event_type",
        "event_timestamp",
        "event_date",
        "event_hour"
    )

In [0]:
dim_product.write.mode("append").format("delta") \
    .save(f"abfss://{GOLD_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/dim_product/")

dim_user.write.mode("append").format("delta") \
    .save(f"abfss://{GOLD_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/dim_user/")

dim_date.write.mode("append").format("delta") \
    .save(f"abfss://{GOLD_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/dim_date/")

dim_region.write.mode("append").format("delta") \
    .save(f"abfss://{GOLD_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/dim_region/")